# Triangle random intersection graphs: run the study

We want to see how the fraction of vertices in the 3-core changes with
community memberships, and how the picture changes as graphs get larger.
This notebook runs three experiments using the same graph model:

1. **Pilot:** scan a broad lambda range and check the running time.
2. **Comparison:** repeat that scan for several increasing graph sizes.
3. **Refinement:** use a finer grid and more trials around the observed rise.

Choose one experiment in **Study settings**, then run all cells using the
project's `.venv` kernel. Graphs are generated independently, with trials
shared across worker processes. Each graph size is saved to its own CSV.
Existing measurements are protected unless you explicitly allow replacement.
Open `04_triangle_analysis.ipynb` afterwards and select the same experiment
there to inspect the saved results.

## What we simulate and measure

Each of the $n$ individuals receives a number of community memberships,
sampled as $D_i \sim \operatorname{Poisson}(\lambda)$. Each community
has three slots, so the total number of memberships must be divisible
by three. If it is not, we discard **all** the sampled counts and draw
a new vector. This samples the Poisson counts conditional on a
divisible total, rather than adjusting one individual's count.
We then randomly match the memberships to community slots.
Different individuals who share a community are connected. Membership
in the same community more than once creates no self-edge, and a pair
of individuals is connected by at most one edge.

The parameter $\lambda$ is the mean of the Poisson membership
distribution before conditioning. It does not directly specify the
number of graph neighbours: each membership can add two neighbours.
For large sparse graphs, the mean graph degree is approximately
$2\lambda$; repeated memberships and shared neighbours affect this
approximation in finite graphs.

For each choice of $(n,\lambda)$, we sample several independent graphs
and measure
$$X_{n,\lambda} = \frac{|\operatorname{Core}_3(G_{n,\lambda})|}{n}.$$
The 3-core is what remains after repeatedly removing individuals with
fewer than three remaining neighbours. Its size is divided by **all**
$n$ individuals, including those with no edges. The algorithm works
on a copy, so the sampled graph is available for other measurements.

In [ ]:
from pathlib import Path

# Find the other notebooks whether we start here or at the project root.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

# Load the graph classes and algorithms; this does not run a simulation.
%run "{NOTEBOOK_DIR}/02_algorithms.ipynb"

In [ ]:
import csv
from numbers import Integral
from time import perf_counter

import numpy as np
from joblib import Parallel, delayed

# Interpret output paths from the project root, wherever the kernel starts.
PROJECT_ROOT = NOTEBOOK_DIR.parent

## Study settings

Start with `stage="pilot"`. Check its plots and running time before selecting
`"comparison"` or `"refinement"`. Only the selected experiment runs; the others
are settings ready for the next step. The suggested graph sizes and repeat
counts are starting choices, not mathematical requirements.

Each experiment lists its graph `sizes`, lambda range, grid size, and number
of independent `repeats` at every point. Lambda is the Poisson membership
parameter. The comparison uses the same grid for every graph size. The
refinement uses more trials and smaller spacing, but its bounds should be
adjusted after inspecting the pilot so that they surround the rise.

`jobs=4` runs up to four trials at once. Use `jobs=1` for a serial run or
`jobs=-1` for all CPUs available to joblib. Parallel work happens inside each
graph size; the sizes themselves run in order so we can save each one as soon
as it finishes. Seeds are assigned before work is sent to the workers.

Files are named `<stage>_n<size>.csv` inside `output_dir`, relative to the
project root. To keep a second study, choose another directory. Repeating a
stage raises an error if any of its files already exist; set `overwrite=True`
only when you intend to replace them.

In [ ]:
config = {
    "stage": "pilot",  # Choose "pilot", "comparison", or "refinement".
    "seed": 2026,  # Starting seed for separate streams for each experiment and size.
    "jobs": 4,  # Parallel trials: 1 is serial, -1 uses all available CPUs.
    "output_dir": "results/runs/triangle_study",
    "overwrite": False,  # Keep existing measurements unless replacement is intended.
    "experiments": {
        "pilot": {
            "sizes": [1000],  # A quick first look before choosing larger graphs.
            "lambda_min": 0.7,
            "lambda_max": 2.5,
            "lambda_count": 19,  # Spacing 0.1, including both endpoints.
            "repeats": 10,
        },
        "comparison": {
            "sizes": [1000, 3000, 10000],  # Compare the fraction, not the raw count.
            "lambda_min": 0.7,
            "lambda_max": 2.5,
            "lambda_count": 41,
            "repeats": 20,
        },
        "refinement": {
            "sizes": [1000, 3000, 10000],
            "lambda_min": 1.0,  # Adjust these bounds after reviewing the pilot.
            "lambda_max": 1.5,
            "lambda_count": 51,  # Spacing 0.01 for a closer view of the rise.
            "repeats": 100,  # More trials help reveal variation near the transition.
        },
    },
}
config

## One trial and a repeatable collection of trials

`triangle_trial` generates one graph and records its 3-core size and fraction.
`simulate_triangle` repeats that trial across a supplied lambda grid. It gives
each trial its own random seed before starting any workers, so changing the
worker count leaves the measurements unchanged. Rows follow the supplied
lambda order and then the trial number (`replicate`, starting from zero).

The study also gives each graph size and experiment a separate random stream.
The pilot, comparison, and refinement therefore provide new samples even
when they include the same lambda. Adding another size does not change the
streams for the sizes already chosen. Changing a lambda grid or repeat count
can change which seed is assigned to a particular trial; every row records
the seed actually used.

With the same code and dependencies, a saved trial seed reproduces the graph
structure and core size. UUID vertex labels are generated separately and can
differ; they have no effect on the measured fraction.

In [ ]:
CSV_FIELDS = ("model", "n", "lambda", "replicate", "seed", "core_size", "core_fraction")


def triangle_trial(n: int, lam: float, replicate: int, seed: int) -> dict:
    """Generate one graph and record its core size, settings, and random seed."""
    if not isinstance(n, Integral) or isinstance(n, bool) or n <= 0:
        raise ValueError("n must be a positive integer")
    rng = np.random.default_rng(seed)
    graph = construct_special_triangle_RIG(n, lam, rng=rng)
    core = construct_k_core(graph, 3)
    core_size = len(core.nodes)
    return {
        "model": "triangle_rig",
        "n": int(n),
        "lambda": float(lam),
        "replicate": replicate,
        "seed": seed,
        "core_size": core_size,
        "core_fraction": core_size / n,
    }


def simulate_triangle(
    n: int, lambdas, repeats: int, seed: int = 2026, jobs: int = 1
) -> list[dict]:
    """Run independent trials at one graph size, optionally in parallel.

    Results follow lambda order and then trial number. Each trial receives
    its seed before workers start, so worker scheduling cannot change it.
    """
    for name, value in (("n", n), ("repeats", repeats), ("seed", seed), ("jobs", jobs)):
        if not isinstance(value, Integral) or isinstance(value, bool):
            raise ValueError(f"{name} must be an integer")
    if n <= 0 or repeats <= 0 or seed < 0 or jobs == 0:
        raise ValueError("Use positive n/repeats, a nonnegative seed, and nonzero jobs")
    grid = [float(lam) for lam in lambdas]
    if (
        not grid
        or len(set(grid)) != len(grid)
        or any(not np.isfinite(lam) or lam < 0 for lam in grid)
    ):
        raise ValueError("Use distinct, finite, nonnegative lambda values")

    # Separate random streams make the trials independent of worker scheduling.
    child_seeds = iter(np.random.SeedSequence(seed).spawn(len(grid) * repeats))
    tasks = []
    for lam in grid:
        for replicate in range(repeats):
            # Keep all seed bits by saving an integer, not a floating-point value.
            trial_seed = int(next(child_seeds).generate_state(1, dtype=np.uint64)[0])
            tasks.append((n, lam, replicate, trial_seed))

    return Parallel(n_jobs=jobs, verbose=5)(
        delayed(triangle_trial)(*task) for task in tasks
    )


def save_results(rows: list[dict], path: str | Path, overwrite: bool = False) -> None:
    """Save the trial rows, protecting an existing CSV unless overwrite is set."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Exclusive creation also protects against another run choosing the same file.
    with path.open("w" if overwrite else "x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=CSV_FIELDS)
        writer.writeheader()
        writer.writerows(rows)


def run_triangle_study(
    config: dict, project_root: str | Path = PROJECT_ROOT
) -> list[Path]:
    """Run the chosen experiment and save one file after each graph size finishes."""
    stage = config["stage"]
    stage_names = ("pilot", "comparison", "refinement")
    if stage not in stage_names or stage not in config["experiments"]:
        raise ValueError("stage must select pilot, comparison, or refinement")
    experiment = config["experiments"][stage]
    sizes = list(experiment["sizes"])
    if (
        not sizes
        or any(
            not isinstance(n, Integral) or isinstance(n, bool) or n <= 0 for n in sizes
        )
        or len(set(sizes)) != len(sizes)
    ):
        raise ValueError("sizes must contain distinct positive integers")
    count = experiment["lambda_count"]
    lower, upper = experiment["lambda_min"], experiment["lambda_max"]
    if not isinstance(count, Integral) or isinstance(count, bool) or count < 3:
        raise ValueError("lambda_count must be an integer of at least three")
    if not np.isfinite(lower) or not np.isfinite(upper) or not 0 <= lower < upper:
        raise ValueError("Require finite bounds with 0 <= lambda_min < lambda_max")
    lambdas = np.linspace(lower, upper, count)
    output_dir = Path(project_root) / config["output_dir"]
    paths = [output_dir / f"{stage}_n{n}.csv" for n in sizes]
    overwrite = config.get("overwrite", False)
    if not overwrite:
        existing = [str(path) for path in paths if path.exists()]
        if existing:
            raise FileExistsError(
                "Measurements already exist: "
                + ", ".join(existing)
                + ". Choose another output_dir or explicitly set overwrite=True."
            )

    total_trials = len(sizes) * count * experiment["repeats"]
    print(
        f"{stage}: {len(sizes)} graph sizes, {count} lambda values, {total_trials} trials."
    )
    for n, path in zip(sizes, paths):
        # Keep experiments and sizes independent, without depending on size-list order.
        size_seed = int(
            np.random.SeedSequence(
                [config["seed"], stage_names.index(stage), int(n)]
            ).generate_state(1, dtype=np.uint64)[0]
        )
        started = perf_counter()
        rows = simulate_triangle(
            n, lambdas, experiment["repeats"], seed=size_seed, jobs=config["jobs"]
        )
        save_results(rows, path, overwrite=overwrite)
        seconds = perf_counter() - started
        print(
            f"n={n}: saved {len(rows)} trials to {path} in {seconds:.1f} s "
            f"({len(rows) / max(seconds, 1e-9):.1f} trials/s)."
        )
    return paths

## Run the selected experiment

This cell does the numerical work. Joblib reports progress as trials finish,
and the notebook prints the elapsed time for each graph size. Use the pilot's
time as a guide before increasing graph sizes, grid resolution, or repeat
counts. Each completed size is saved immediately, so its measurements remain
available even if a later size is interrupted.

Every CSV row contains the model, `n`, lambda, trial number, seed, core size,
and core fraction. Keep these individual measurements: an average alone can
hide a mixture of graphs with almost empty and substantial cores.

In [ ]:
data_files = run_triangle_study(config)
data_files

## What to do after a run

In notebook 04, select the same `stage` and data directory. Inspect the broad
curves first, then compare graph sizes and look at individual outcomes near
the rise. Check whether the apparent location changes with graph size, lambda
spacing, or smoothing before interpreting it as evidence about a transition.

After the pilot, adjust the refinement bounds to surround the observed rise.
For a new study, change `output_dir` in this notebook and `data_dir` in the
analysis notebook. A finite collection of simulations can support a conclusion
about a jump or gradual emergence; it does not prove the limiting behaviour.